In [0]:
import pyspark

In [0]:
df_airports = (
    spark.read
    .option("header", "true")
    .option("delimiter", ",")
    .csv("/Volumes/airtravel/airtravelschema/airport/airports.csv")
)


In [0]:
df_runways = (
    spark.read
    .option("header", "true")
    .option("delimiter", ",")
    .csv("/Volumes/airtravel/airtravelschema/airport/runways.csv")
)

In [0]:
from pyspark.sql.functions import col
df_airports = (
    df_airports
    .withColumn("latitude_deg", col("latitude_deg").cast("double"))
    .withColumn("longitude_deg", col("longitude_deg").cast("double"))
    .withColumn("elevation_ft", col("elevation_ft").cast("int"))
)

In [0]:
df_runways = (
    df_runways
    .withColumn("length_ft", col("length_ft").cast("int"))
    .withColumn("width_ft", col("width_ft").cast("int"))
)

In [0]:
print("AIRPORTS DATA")
df_airports.printSchema()
df_airports.show(5)

print("RUNWAYS DATA")
df_runways.printSchema()
df_runways.show(5)

AIRPORTS DATA
root
 |-- id: string (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)

+------+-----+-------------+--------------------+------------+-------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+

In [0]:
# Print row count

print("Number of rows in df_airports:", df_airports.count())

print("Number of rows in df_runways:", df_runways.count())

Number of rows in df_airports: 85324
Number of rows in df_runways: 47884


In [0]:
# Inner join between df_airports and df_runways

df_combined = (
    df_airports.join(
        df_runways,
        df_airports.ident == df_runways.airport_ident,
        "inner"
    )
)

# View schema and sample data
df_combined.printSchema()
df_combined.show(5)

root
 |-- id: string (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- id: string (nullable = true)
 |-- airport_ref: string (nullable = true)
 |-- airport_ident: string (nullable = true)
 |-- length_ft: integer (nullable = true)
 |-- width_ft: integer (nullable = true)
 |-

In [0]:
from pyspark.sql.functions import when, col

# Add runway_category column
df_combined = (
    df_combined.withColumn(
        "runway_category",
        when(col("length_ft") > 10000, "Long")
        .when((col("length_ft") >= 5000) & (col("length_ft") <= 9999), "Medium")
        .otherwise("Short")
    )
)

# Verify result
df_combined.select("length_ft", "runway_category").show(10)

+---------+---------------+
|length_ft|runway_category|
+---------+---------------+
|       80|          Short|
|     2500|          Short|
|     2100|          Short|
|     4517|          Short|
|     1450|          Short|
|     1700|          Short|
|     6000|         Medium|
|     3200|          Short|
|     4700|          Short|
|     2600|          Short|
+---------+---------------+
only showing top 10 rows


In [0]:
df_combined.createOrReplaceTempView("airports_view")

In [0]:
%sql
SELECT 
iso_country,
COUNT(*) AS runway_count
FROM airports_view
GROUP BY iso_country
ORDER BY runway_count DESC
LIMIT 10;

iso_country,runway_count
US,26450
BR,5825
AU,2103
CA,1578
AR,761
FR,659
GB,600
DE,592
RU,489
ID,414


In [0]:
spark.sql("""
SELECT 
    continent,
    AVG(length_ft) AS avg_runway_length
FROM airports_view
WHERE length_ft IS NOT NULL
GROUP BY continent
ORDER BY avg_runway_length DESC
""").show()

+---------+------------------+
|continent| avg_runway_length|
+---------+------------------+
|       AN| 7768.857142857143|
|       AS|6925.6469996647675|
|       AF| 6787.526448362721|
|       EU| 4103.151335311572|
|       OC|3558.0401785714284|
|       SA|2857.0891141239963|
|       NA| 2599.724218941758|
+---------+------------------+



In [0]:
spark.sql("""
SELECT 
    name,
    municipality,
    COUNT(*) AS lighted_runway_count
FROM airports_view
WHERE iso_country = 'IN'
  AND lighted = '1'
GROUP BY name, municipality
HAVING COUNT(*) >= 1
""").show()

+--------------------+------------------+--------------------+
|                name|      municipality|lighted_runway_count|
+--------------------+------------------+--------------------+
|Maharshi Valmiki ...|          Faizabad|                   1|
|Navi Mumbai Inter...|       Navi Mumbai|                   2|
|Dholera Internati...|           Dholera|                   1|
|     Mehsana Airport|           Mehsana|                   1|
|      Dhulia Airport|              NULL|                   1|
|Raigarh Airport (...|              NULL|                   1|
|   Khajuraho Airport|         Khajuraho|                   1|
|     Dimapur Airport|           Dimapur|                   1|
|      Purnea Airport|              NULL|                   1|
|Hindon Airport / ...|         Ghaziabad|                   1|
|     Jodhpur Airport|           Jodhpur|                   1|
|       Jammu Airport|             Jammu|                   1|
|   Shravasti Airport|         Shravasti|              

In [0]:
spark.sql("""
SELECT 
    name,
    iso_country,
    MAX(length_ft) AS max_runway_length
FROM airports_view
WHERE length_ft IS NOT NULL
GROUP BY name, iso_country
ORDER BY max_runway_length DESC
LIMIT 5
""").show()

+--------------------+-----------+-----------------+
|                name|iso_country|max_runway_length|
+--------------------+-----------+-----------------+
|Gunflint Seaplane...|         US|            30000|
|Libby Camps Seapl...|         US|            26000|
|Brookville Reserv...|         US|            25000|
|Long Lake Seaplan...|         US|            25000|
|Conchas Lake Seap...|         US|            21120|
+--------------------+-----------+-----------------+



In [0]:
# Join with renamed duplicate columns

df_combined = (
    df_airports.alias("a")
    .join(
        df_runways.alias("r"),
        col("a.ident") == col("r.airport_ident"),
        "inner"
    )
    .select(
        col("a.id").alias("airport_id"),
        col("a.ident"),
        col("a.type"),
        col("a.name"),
        col("a.latitude_deg"),
        col("a.longitude_deg"),
        col("a.elevation_ft"),
        col("a.continent"),
        col("a.iso_country"),
        col("a.municipality"),

        col("r.id").alias("runway_id"),
        col("r.airport_ident"),
        col("r.length_ft"),
        col("r.width_ft"),
        col("r.lighted")
    )
)

In [0]:
from pyspark.sql.functions import when

df_combined = (
    df_combined.withColumn(
        "runway_category",
        when(col("length_ft") > 10000, "Long")
        .when((col("length_ft") >= 5000) & (col("length_ft") <= 9999), "Medium")
        .otherwise("Short")
    )
)

In [0]:
df_combined.write \
    .mode("overwrite") \
    .parquet("/Volumes/airtravel/airtravelschema/airport/combined_parquet")

In [0]:
# Read the saved Parquet file

df_parquet = spark.read.parquet(
    "/Volumes/airtravel/airtravelschema/airport/combined_parquet/"
)

# Display first 10 rows
df_parquet.show(10)

+----------+-----+-------------+--------------------+------------+-------------+------------+---------+-----------+------------+---------+-------------+---------+--------+-------+---------------+
|airport_id|ident|         type|                name|latitude_deg|longitude_deg|elevation_ft|continent|iso_country|municipality|runway_id|airport_ident|length_ft|width_ft|lighted|runway_category|
+----------+-----+-------------+--------------------+------------+-------------+------------+---------+-----------+------------+---------+-------------+---------+--------+-------+---------------+
|      6523|  00A|     heliport|   Total RF Heliport|   40.070985|   -74.933689|          11|       NA|         US|    Bensalem|   269408|          00A|       80|      80|      1|          Short|
|      6524| 00AK|small_airport|        Lowell Field|   59.947733|  -151.692524|         450|       NA|         US|Anchor Point|   255155|         00AK|     2500|      40|      0|          Short|
|      6525| 00AL|sm